In [1]:
# Canadian Stocks CNR, CSU, DOL: K-Means + NLP

%pip install -q pandas numpy matplotlib scikit-learn nltk

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)

# 1. Load dataset
df = pd.read_csv("Canadian_Stocks_CNR_CSU_DOL.csv")
df.columns = df.columns.str.strip()
print(df.head())

# 2. K-Means clustering
X = df.select_dtypes(include=np.number).copy()
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all")
X = X.fillna(X.median())
X = X.loc[:, X.nunique() > 1]

if len(X) < 3 or X.shape[1] == 0:
    raise ValueError("Insufficient numeric data for clustering.")

X_scaled = StandardScaler().fit_transform(X)
k = min(3, len(X) - 1)

df.loc[X.index, "Cluster"] = KMeans(
    n_clusters=k, random_state=42, n_init=10
).fit_predict(X_scaled)

# 3. Plot clusters
points = PCA(n_components=2).fit_transform(X_scaled)
plt.scatter(points[:, 0], points[:, 1],
            c=df.loc[X.index, "Cluster"], cmap="viridis")
plt.title("K-Means Clustering: CNR, CSU, DOL")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

# 4. NLP sentiment analysis (requires a news/text column)
text_col = next(
    (c for c in df.columns if any(
        word in c.lower()
        for word in ["news", "headline", "text", "article", "title"]
    )),
    None
)

if text_col:
    sia = SentimentIntensityAnalyzer()
    df["Sentiment_Score"] = df[text_col].fillna("").astype(str).apply(
        lambda text: sia.polarity_scores(text)["compound"]
    )
    df["Sentiment"] = df["Sentiment_Score"].apply(
        lambda score: "Positive" if score >= 0.05
        else "Negative" if score <= -0.05
        else "Neutral"
    )
    print(df["Sentiment"].value_counts())
else:
    print("NLP skipped: no news or text column found.")

# 5. Save results
df.to_csv("Canadian_Stocks_CNR_CSU_DOL_Results.csv", index=False)
print("Analysis complete. Results saved.")

Note: you may need to restart the kernel to use updated packages.
                             Company  Ticker(s)    Exchange           Sector  \
0  Canadian National Railway Company  CNR / CNI  TSX / NYSE   Transportation   
1        Constellation Software Inc.        CSU         TSX       Technology   
2                     Dollarama Inc.        DOL         TSX  Consumer Retail   

                                Business description  \
0  Freight railroad network linking Canada’s coas...   
1  Acquires and operates vertical-market software...   
2  Canada’s leading value and discount retailer w...   

           Latest quoted price Price currency  Price date  \
0  168.06 (TSX); 117.78 (NYSE)      CAD / USD  2026-10-09   
1                      3003.78            CAD  2026-10-09   
2                       191.93            CAD  2026-10-09   

                                Investment character  
0  Transportation infrastructure; cyclical freigh...  
1  High-growth software compounde

ValueError: Insufficient numeric data for clustering.